In [1]:
import random_matrix.amplitude_matrix.isotropic_tmatrix as isotropic_tmatrix
from scipy.special import hankel1, spherical_jn, lpmv, h1vp,spherical_yn
import numpy as np
import math

1. Generating the T_matrix


In [2]:
hull_uniform = isotropic_tmatrix.create_hull_uniform() # Creates the Convex hull for a sphere of size 600nm

wavelength1 = 532e-9
k1 = (2 * np.pi) / wavelength1
m = 1.5 # Relative refractive Index
k2 = m * k1
n_max = 2

T_uniform= isotropic_tmatrix.get_T(hull_uniform,  k1, k2, n_max)


Number of simplices: 4802
Number of modes: 8 


  0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:06<00:00,  1.30it/s]


In [3]:
T_uniform[1][1]

(-0.18519673563717184+0.38845708224526854j)

In [5]:
hull_inv_transform = isotropic_tmatrix.create_hull_inv_transform() # Creates the Convex hull for a sphere of size 600nm

wavelength1 = 532e-9
k1 = (2 * np.pi) / wavelength1
m = 1.5 # Relative refractive Index
k2 = m * k1
n_max = 2

T_inv_transform= isotropic_tmatrix.get_T(hull_inv_transform,  k1, k2, n_max)

Number of simplices: 4802
Number of modes: 8 


100%|██████████| 8/8 [00:10<00:00,  1.30s/it]


In [9]:
true_area = 4*np.pi*(600e-9)**2
print("Error in area for uniform sampled sphere",np.abs(hull_uniform.area-true_area))
print("Error in area for inverse transformed sampled sphere",np.abs(hull_inv_transform.area-true_area))

Error in area for uniform sampled sphere 8.42380993504889e-15
Error in area for inverse transformed sampled sphere 9.77041606994078e-15


2. Functions that calculate required constants.

In [4]:

def alph_mn_mpnp(m,n,mpr,npr):
    
    return (-1)**(m+mpr)*(1j)**(npr-n-1)*np.sqrt(((2*n+1)*(2*npr+1)/(n*(n+1)*npr*(npr+1))))

def h_n(n, kr):
    return spherical_jn(n, kr) + 1j * spherical_yn(n, kr)


def dh_n(n, x):
    """Computes the derivative of the spherical Hankel function of the first kind."""
    hn = h_n(n, x)
    hn1 = h_n(n + 1, x)
    return (n / x) * hn - hn1

def b_n(n, m, x):
    psi_n_x = x*spherical_jn(n, x)
    psi_n_mx = m*x*spherical_jn(n, m * x)
    psi_n_x_derivative = x*spherical_jn(n, x, derivative=True)+spherical_jn(n,x)
    psi_n_mx_derivative = (m*x*spherical_jn(n, m * x, derivative=True)+spherical_jn(n,m*x))
    xi_n_x = x*h_n(n, x)
    xi_n_x_derivative = x*dh_n(n, x)+h_n(n,x)
    
    num = m * psi_n_x * psi_n_mx_derivative - psi_n_mx * psi_n_x_derivative
    den = m * xi_n_x * psi_n_mx_derivative - psi_n_mx * xi_n_x_derivative
    
    return num / den




In [5]:
wavelength1 = 532e-9
k1 = (2 * np.pi) / wavelength1
r =600e-9
x = k1*r
rri = 1.5
n=1

print("n=1")
bn = b_n(n,1.5,x)
print("Analytical Mie theory b_n value:",b_n(n,1.5,x))
computed_b_n = -1j*(n*(n+1)/(2*n+1))*alph_mn_mpnp(0,1,0,1)*T_uniform[1][1]
print("Computed b_n vlaues from T matrix",computed_b_n)
print("Error:",np.abs(computed_b_n-bn))


n=1
Analytical Mie theory b_n value: (0.18554725921886245-0.38874088261363854j)
Computed b_n vlaues from T matrix (0.18519673563717184-0.38845708224526854j)
Error: 0.00045100934625366495


In [6]:
wavelength1 = 532e-9
k1 = (2 * np.pi) / wavelength1
r =600e-9
x = k1*r
rri = 1.5
n=1
print("n=1")
bn = b_n(n,1.5,x)
print("Analytical Mie theory b_n value:",b_n(n,1.5,x))
computed_b_n = -1j*(n*(n+1)/(2*n+1))*alph_mn_mpnp(0,1,0,1)*T_inv_transform[1][1]
print("Computed b_n vlaues from T matrix",computed_b_n)
print("Error:",np.abs(computed_b_n-bn))

n=1
Analytical Mie theory b_n value: (0.18554725921886245-0.38874088261363854j)


NameError: name 'T_inv_transform' is not defined

In [ ]:
# @staticmethod
def integration_check(n1, m1, n2, m2):
    wavelength1 = 532e-9
    k1 = (2 * np.pi) / wavelength1
    wavelength2 = 266e-9
    k2 = (2 * np.pi) / wavelength2
    row = 50
    col = 51
    num_linear = row * col
    theta = np.linspace(0, np.pi, col)
    phi = np.linspace(0, 2 * np.pi, row)
    theta_grid, phi_grid = np.meshgrid(theta, phi)
    r = 10e-9

    def integrand1(theta, phi):

        RgM = RgM_mn(0, 1, k2 * r, theta, phi)
        N = N_mn(0, 1, k1 * r, theta, phi)
        result = (RgM[0] * N[2] - RgM[1] * N[1]) * r**2 * np.sin(theta)

        return result

    def integrand2(theta, phi):
        B1 = B_mn(m1, n1, theta, phi)
        B2 = B_mn(m2, n2, theta, phi)
        return (B1[0] * np.conj(B2[0]) + B1[1] * np.conj(B2[1])) * np.sin(theta)

    def integrand3(theta, phi):
        C1 = C_mn(m1, n1, theta, phi)
        C2 = C_mn(m2, n2, theta, phi)
        return (C1[0] * np.conj(C2[0]) + C1[1] * np.conj(C2[1])) * np.sin(theta)

    def integrand4(theta, phi):
        M1 = M_mn(1, 1, k1, theta, phi)
        M2 = M_mn(1, 1, k1, theta, phi)
        return (M1[0] * np.conj(M2[0]) + M1[1] * np.conj(M2[1])) * np.sin(theta)

    def integrand5(theta, phi):
        P1 = P_mn(m1, n1, theta, phi)
        P2 = P_mn(m2, n2, theta, phi)
        return P1 * np.conj(P2) * np.sin(theta)

        # Compute the integral over [-1, 1]
        # n = 1
        # m = 1

        # gamma_mn = np.sqrt(
        ((2 * n + 1) * factorial(n - m)) / (4 * np.pi * n * (n + 1) * factorial(n + m))

    # )
    result, _ = dblquad(integrand3, 0, 2 * np.pi, 0, np.pi)
    # r = result / (h_n(n, k) * np.conj(h_n(n, k)))
    return result


"""
hull, points = create_hull()
wavelength1 = 532e-9
k1 = (2 * np.pi) / wavelength1
wavelength2 = 266e-9
k2 = 1.5 * k1
n_max = 2
modes = sum(2 * n + 1 for n in range(1, n_max + 1))
print(f"Total number of modes is {modes}")
T = get_T(hull, points, k1, k2, n_max)
plt.matshow(np.abs(T), cmap="viridis")
plt.title("T Matrix")
plt.colorbar()
plt.show()
"""


n=1  
Sampling used : 100 uniform phi points and 101 uniform theta point  
Analytical Mie theory b_n value: (-0.05043054394533954-0.19384157896230952j)  
Computed b_n vlaues from T matrix (-0.050229944914698905-0.1949243621541355j)

Sampling used : 50 uniform phi points and 51 uniform theta point  
Analytical Mie theory b_n value: (-0.05043054394533954-0.19384157896230952j)  
Computed b_n vlaues from T matrix (-0.05021801181055005-0.1947708794468376j)